# Supply Chain Data Warehouse — EDA
**10 Tables | 35 Questions**

Tables: `dim_customer`, `dim_date`, `dim_facility`, `dim_product`, `dim_supplier`, `fact_inventory`, `fact_procurement`, `fact_production`, `fact_sales`, `fact_shipment`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Generate synthetic data ──────────────────────────────────────────────────
np.random.seed(42)
n = 500

# dim_date
dates = pd.date_range('2022-01-01', periods=730)
dim_date = pd.DataFrame({
    'date_key':    range(1, 731),
    'date':        dates,
    'year':        dates.year,
    'quarter':     dates.quarter,
    'month':       dates.month,
    'month_name':  dates.month_name(),
    'week':        dates.isocalendar().week.values,
    'day':         dates.day,
    'day_of_week': dates.dayofweek,
    'day_name':    dates.day_name(),
    'is_weekend':  dates.dayofweek >= 5
})

# dim_customer
dim_customer = pd.DataFrame({
    'customer_id':        range(1, 101),
    'customer_key':       ['CK' + str(i).zfill(4) for i in range(1, 101)],
    'customer_name':      ['Customer_' + str(i) for i in range(1, 101)],
    'country':            np.random.choice(['USA', 'Germany', 'China', 'UK', 'India'], 100),
    'channel_type':       np.random.choice(['Retail', 'Wholesale', 'Online', 'Direct'], 100),
    'size':               np.random.choice(['Small', 'Medium', 'Large', 'Enterprise'], 100),
    'annual_volume_usd':  np.random.randint(50000, 5000000, 100)
})

# dim_facility
dim_facility = pd.DataFrame({
    'facility_id':      range(1, 21),
    'facility_key':     ['FK' + str(i).zfill(3) for i in range(1, 21)],
    'facility_name':    ['Facility_' + str(i) for i in range(1, 21)],
    'country':          np.random.choice(['USA', 'Germany', 'China', 'Mexico', 'India'], 20),
    'city':             np.random.choice(['New York', 'Berlin', 'Shanghai', 'CDMX', 'Mumbai'], 20),
    'facility_type':    np.random.choice(['Warehouse', 'Plant', 'Distribution Center'], 20),
    'specialization':   np.random.choice(['Electronics', 'Automotive', 'Consumer Goods', 'Pharma'], 20),
    'annual_capacity':  np.random.randint(10000, 500000, 20)
})

# dim_product
dim_product = pd.DataFrame({
    'product_id':   range(1, 51),
    'product_key':  ['PK' + str(i).zfill(4) for i in range(1, 51)],
    'product_name': ['Product_' + str(i) for i in range(1, 51)],
    'category':     np.random.choice(['Electronics', 'Clothing', 'Home Goods', 'Tools', 'Food'], 50),
    'product_line': np.random.choice(['Budget', 'Standard', 'Premium', 'Luxury'], 50),
    'spec':         np.random.choice(['Spec_A', 'Spec_B', 'Spec_C'], 50),
    'color':        np.random.choice(['Red', 'Blue', 'Black', 'White', 'Green'], 50),
    'unit_price':   np.round(np.random.uniform(5, 500, 50), 2),
    'unit_cost':    np.round(np.random.uniform(2, 300, 50), 2),
    'weight_kg':    np.round(np.random.uniform(0.1, 50, 50), 2)
})

# dim_supplier
dim_supplier = pd.DataFrame({
    'supplier_id':       range(1, 31),
    'supplier_key':      ['SK' + str(i).zfill(3) for i in range(1, 31)],
    'supplier_name':     ['Supplier_' + str(i) for i in range(1, 31)],
    'country':           np.random.choice(['USA', 'China', 'Germany', 'Japan', 'India'], 30),
    'city':              np.random.choice(['Chicago', 'Beijing', 'Hamburg', 'Tokyo', 'Delhi'], 30),
    'specialty':         np.random.choice(['Raw Materials', 'Components', 'Packaging', 'Chemicals'], 30),
    'tier':              np.random.choice([1, 2, 3], 30),
    'avg_quality_score': np.round(np.random.uniform(60, 100, 30), 1)
})

# fact_sales
fact_sales = pd.DataFrame({
    'sales_id':          range(1, n+1),
    'date_key':          np.random.randint(1, 731, n),
    'product_id':        np.random.randint(1, 51, n),
    'customer_id':       np.random.randint(1, 101, n),
    'quantity_sold':     np.random.randint(1, 200, n),
    'unit_price':        np.round(np.random.uniform(5, 500, n), 2),
    'discount_pct':      np.round(np.random.uniform(0, 0.3, n), 2),
    'order_number':      ['ORD' + str(i).zfill(6) for i in range(1, n+1)]
})
fact_sales['discount_amount'] = fact_sales['unit_price'] * fact_sales['quantity_sold'] * fact_sales['discount_pct']
fact_sales['gross_revenue']   = fact_sales['unit_price'] * fact_sales['quantity_sold']
fact_sales['net_revenue']     = fact_sales['gross_revenue'] - fact_sales['discount_amount']
fact_sales['total_cost']      = fact_sales['unit_price'] * 0.6 * fact_sales['quantity_sold']
fact_sales['profit']          = fact_sales['net_revenue'] - fact_sales['total_cost']
fact_sales['profit_margin_pct'] = (fact_sales['profit'] / fact_sales['net_revenue'] * 100).round(2)

# fact_inventory
fact_inventory = pd.DataFrame({
    'inventory_id':      range(1, n+1),
    'date_key':          np.random.randint(1, 731, n),
    'product_id':        np.random.randint(1, 51, n),
    'facility_id':       np.random.randint(1, 21, n),
    'stock_level':       np.random.randint(0, 5000, n),
    'safety_stock_level':np.random.randint(100, 1000, n),
    'reorder_point':     np.random.randint(200, 2000, n)
})

# fact_procurement
fact_procurement = pd.DataFrame({
    'procurement_id':  range(1, n+1),
    'order_date_key':  np.random.randint(1, 731, n),
    'product_id':      np.random.randint(1, 51, n),
    'supplier_id':     np.random.randint(1, 31, n),
    'order_quantity':  np.random.randint(50, 1000, n),
    'unit_cost':       np.round(np.random.uniform(2, 300, n), 2),
    'lead_time_days':  np.random.randint(1, 60, n),
    'delivery_date_key':np.random.randint(1, 731, n),
    'quality_score':   np.round(np.random.uniform(60, 100, n), 1),
    'po_number':       ['PO' + str(i).zfill(6) for i in range(1, n+1)]
})
fact_procurement['total_cost'] = fact_procurement['order_quantity'] * fact_procurement['unit_cost']

# fact_production
fact_production = pd.DataFrame({
    'production_id':    range(1, n+1),
    'date_key':         np.random.randint(1, 731, n),
    'product_id':       np.random.randint(1, 51, n),
    'facility_id':      np.random.randint(1, 21, n),
    'quantity_produced':np.random.randint(100, 5000, n),
    'defective_units':  np.random.randint(0, 200, n),
    'batch_number':     ['BT' + str(i).zfill(6) for i in range(1, n+1)]
})
fact_production['defect_rate_pct'] = (fact_production['defective_units'] / fact_production['quantity_produced'] * 100).round(2)

# fact_shipment
fact_shipment = pd.DataFrame({
    'shipment_id':     range(1, n+1),
    'ship_date_key':   np.random.randint(1, 731, n),
    'delivery_date_key':np.random.randint(1, 731, n),
    'product_id':      np.random.randint(1, 51, n),
    'facility_id':     np.random.randint(1, 21, n),
    'customer_id':     np.random.randint(1, 101, n),
    'quantity':        np.random.randint(1, 200, n),
    'carrier':         np.random.choice(['DHL', 'FedEx', 'UPS', 'Maersk', 'Amazon Logistics'], n),
    'status':          np.random.choice(['Delivered', 'In Transit', 'Delayed', 'Returned'], n, p=[0.6, 0.2, 0.15, 0.05]),
    'shipping_cost':   np.round(np.random.uniform(10, 500, n), 2),
    'total_weight_kg': np.round(np.random.uniform(0.5, 100, n), 2),
    'tracking_number': ['TRK' + str(i).zfill(8) for i in range(1, n+1)],
    'delay_reason':    np.random.choice(['None', 'Weather', 'Customs', 'Carrier Delay', 'Address Issue'], n)
})

print('All tables created!')
print('fact_sales shape      :', fact_sales.shape)
print('fact_inventory shape  :', fact_inventory.shape)
print('fact_procurement shape:', fact_procurement.shape)
print('fact_production shape :', fact_production.shape)
print('fact_shipment shape   :', fact_shipment.shape)

---
## Section 1 — Dimension Tables Overview
### Q1. What does each dimension table look like? (head + shape)

In [ ]:
dims = {'dim_customer': dim_customer, 'dim_date': dim_date,
        'dim_facility': dim_facility, 'dim_product': dim_product,
        'dim_supplier': dim_supplier}

for name, df in dims.items():
    print(f'\n=== {name} | shape: {df.shape} ===')
    print(df.head(3))

### Q2. How many unique customers are there per country?

In [ ]:
country_counts = dim_customer['country'].value_counts()
print(country_counts)

country_counts.plot(kind='bar', title='Customers per Country')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Q3. What is the distribution of customer sizes?

In [ ]:
size_counts = dim_customer['size'].value_counts()
print(size_counts)

size_counts.plot(kind='pie', autopct='%1.0f%%', title='Customer Size Distribution')
plt.ylabel('')
plt.tight_layout()
plt.show()

### Q4. What are the top 10 customers by annual volume?

In [ ]:
top10 = dim_customer.nlargest(10, 'annual_volume_usd')[['customer_name', 'annual_volume_usd']]
print(top10)

plt.barh(top10['customer_name'], top10['annual_volume_usd'])
plt.xlabel('Annual Volume USD')
plt.title('Top 10 Customers by Annual Volume')
plt.tight_layout()
plt.show()

### Q5. What product categories exist and how many products are in each?

In [ ]:
cat_counts = dim_product['category'].value_counts()
print(cat_counts)

cat_counts.plot(kind='bar', title='Products per Category')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Q6. What is the average unit price and unit cost per product category?

In [ ]:
price_cost = dim_product.groupby('category')[['unit_price', 'unit_cost']].mean().round(2)
print(price_cost)

price_cost.plot(kind='bar', title='Avg Price vs Cost per Category')
plt.ylabel('USD')
plt.tight_layout()
plt.show()

### Q7. How many suppliers are there per tier and per country?

In [ ]:
print('--- Suppliers per Tier ---')
print(dim_supplier['tier'].value_counts())

print('\n--- Suppliers per Country ---')
print(dim_supplier['country'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
dim_supplier['tier'].value_counts().plot(kind='bar', ax=axes[0], title='Suppliers by Tier')
dim_supplier['country'].value_counts().plot(kind='bar', ax=axes[1], title='Suppliers by Country')
plt.tight_layout()
plt.show()

---
## Section 2 — Sales Analysis
### Q8. What is the total gross revenue, net revenue, and profit overall?

In [ ]:
total_gross  = fact_sales['gross_revenue'].sum()
total_net    = fact_sales['net_revenue'].sum()
total_profit = fact_sales['profit'].sum()

print(f'Gross Revenue : ${total_gross:,.2f}')
print(f'Net Revenue   : ${total_net:,.2f}')
print(f'Total Profit  : ${total_profit:,.2f}')

plt.bar(['Gross Revenue', 'Net Revenue', 'Total Profit'], [total_gross, total_net, total_profit])
plt.ylabel('USD')
plt.title('Overall Revenue & Profit')
plt.tight_layout()
plt.show()

### Q9. What is the monthly net revenue trend?

In [ ]:
sales_date = fact_sales.merge(dim_date[['date_key', 'year', 'month']], on='date_key')
monthly_rev = sales_date.groupby(['year', 'month'])['net_revenue'].sum().reset_index()
monthly_rev['period'] = monthly_rev['year'].astype(str) + '-' + monthly_rev['month'].astype(str).str.zfill(2)
monthly_rev = monthly_rev.sort_values('period')

plt.plot(monthly_rev['period'], monthly_rev['net_revenue'], marker='o')
plt.xticks(rotation=90)
plt.title('Monthly Net Revenue')
plt.ylabel('Net Revenue USD')
plt.tight_layout()
plt.show()

### Q10. Which product category generates the most profit?

In [ ]:
sales_prod = fact_sales.merge(dim_product[['product_id', 'category']], on='product_id')
cat_profit = sales_prod.groupby('category')['profit'].sum().sort_values(ascending=False)
print(cat_profit)

cat_profit.plot(kind='bar', title='Total Profit by Product Category')
plt.ylabel('Profit USD')
plt.tight_layout()
plt.show()

### Q11. What is the average profit margin per product line?

In [ ]:
sales_pl = fact_sales.merge(dim_product[['product_id', 'product_line']], on='product_id')
margin_pl = sales_pl.groupby('product_line')['profit_margin_pct'].mean().round(2).sort_values(ascending=False)
print(margin_pl)

margin_pl.plot(kind='bar', title='Avg Profit Margin % by Product Line')
plt.ylabel('Margin %')
plt.tight_layout()
plt.show()

### Q12. Which channel type brings in the highest net revenue?

In [ ]:
sales_cust = fact_sales.merge(dim_customer[['customer_id', 'channel_type']], on='customer_id')
channel_rev = sales_cust.groupby('channel_type')['net_revenue'].sum().sort_values(ascending=False)
print(channel_rev)

channel_rev.plot(kind='bar', title='Net Revenue by Channel Type')
plt.ylabel('Net Revenue USD')
plt.tight_layout()
plt.show()

### Q13. What is the distribution of discount percentages applied on sales?

In [ ]:
plt.hist(fact_sales['discount_pct'] * 100, bins=20)
plt.xlabel('Discount %')
plt.ylabel('Frequency')
plt.title('Distribution of Discount Percentages')
plt.tight_layout()
plt.show()

print(fact_sales['discount_pct'].describe())

### Q14. Which top 5 customers have the highest total net revenue?

In [ ]:
sales_cname = fact_sales.merge(dim_customer[['customer_id', 'customer_name']], on='customer_id')
top5_cust = sales_cname.groupby('customer_name')['net_revenue'].sum().nlargest(5)
print(top5_cust)

top5_cust.plot(kind='bar', title='Top 5 Customers by Net Revenue')
plt.ylabel('Net Revenue USD')
plt.tight_layout()
plt.show()

---
## Section 3 — Inventory Analysis
### Q15. How many inventory records have stock below the safety stock level?

In [ ]:
below_safety = fact_inventory[fact_inventory['stock_level'] < fact_inventory['safety_stock_level']]
print(f'Records below safety stock : {len(below_safety)}')
print(f'Total inventory records    : {len(fact_inventory)}')
print(f'Percentage                 : {len(below_safety)/len(fact_inventory)*100:.1f}%')

### Q16. What is the average stock level per facility?

In [ ]:
inv_fac = fact_inventory.merge(dim_facility[['facility_id', 'facility_name']], on='facility_id')
avg_stock = inv_fac.groupby('facility_name')['stock_level'].mean().sort_values(ascending=False)
print(avg_stock)

avg_stock.plot(kind='bar', title='Avg Stock Level per Facility')
plt.ylabel('Avg Stock')
plt.tight_layout()
plt.show()

### Q17. Which products are most frequently below their reorder point?

In [ ]:
below_reorder = fact_inventory[fact_inventory['stock_level'] < fact_inventory['reorder_point']]
below_reorder_prod = below_reorder.merge(dim_product[['product_id', 'product_name']], on='product_id')
top_reorder = below_reorder_prod['product_name'].value_counts().head(10)
print(top_reorder)

top_reorder.plot(kind='bar', title='Products Most Often Below Reorder Point')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Q18. What is the stock level distribution across all facilities?

In [ ]:
plt.hist(fact_inventory['stock_level'], bins=30)
plt.xlabel('Stock Level')
plt.ylabel('Frequency')
plt.title('Stock Level Distribution')
plt.tight_layout()
plt.show()

print(fact_inventory['stock_level'].describe())

---
## Section 4 — Procurement Analysis
### Q19. What is the total procurement spend per supplier?

In [ ]:
proc_sup = fact_procurement.merge(dim_supplier[['supplier_id', 'supplier_name']], on='supplier_id')
spend_sup = proc_sup.groupby('supplier_name')['total_cost'].sum().sort_values(ascending=False).head(10)
print(spend_sup)

spend_sup.plot(kind='bar', title='Top 10 Suppliers by Procurement Spend')
plt.ylabel('Total Cost USD')
plt.tight_layout()
plt.show()

### Q20. What is the average lead time per supplier tier?

In [ ]:
proc_tier = fact_procurement.merge(dim_supplier[['supplier_id', 'tier']], on='supplier_id')
avg_lead = proc_tier.groupby('tier')['lead_time_days'].mean().round(1)
print(avg_lead)

avg_lead.plot(kind='bar', title='Avg Lead Time by Supplier Tier')
plt.ylabel('Lead Time (days)')
plt.tight_layout()
plt.show()

### Q21. What is the quality score distribution from all procurement orders?

In [ ]:
plt.hist(fact_procurement['quality_score'], bins=20)
plt.xlabel('Quality Score')
plt.ylabel('Frequency')
plt.title('Procurement Quality Score Distribution')
plt.tight_layout()
plt.show()

print(fact_procurement['quality_score'].describe())

### Q22. Which product has the highest total procurement cost?

In [ ]:
proc_prod = fact_procurement.merge(dim_product[['product_id', 'product_name']], on='product_id')
top_proc_prod = proc_prod.groupby('product_name')['total_cost'].sum().nlargest(10)
print(top_proc_prod)

top_proc_prod.plot(kind='bar', title='Top 10 Products by Procurement Cost')
plt.ylabel('Total Cost USD')
plt.tight_layout()
plt.show()

---
## Section 5 — Production Analysis
### Q23. What is the total quantity produced per facility?

In [ ]:
prod_fac = fact_production.merge(dim_facility[['facility_id', 'facility_name']], on='facility_id')
total_produced = prod_fac.groupby('facility_name')['quantity_produced'].sum().sort_values(ascending=False)
print(total_produced)

total_produced.plot(kind='bar', title='Total Quantity Produced per Facility')
plt.ylabel('Units Produced')
plt.tight_layout()
plt.show()

### Q24. Which facility has the highest average defect rate?

In [ ]:
defect_fac = prod_fac.groupby('facility_name')['defect_rate_pct'].mean().sort_values(ascending=False)
print(defect_fac)

defect_fac.plot(kind='bar', title='Avg Defect Rate % per Facility')
plt.ylabel('Defect Rate %')
plt.tight_layout()
plt.show()

### Q25. What is the overall defect rate trend over time (monthly)?

In [ ]:
prod_date = fact_production.merge(dim_date[['date_key', 'year', 'month']], on='date_key')
monthly_defect = prod_date.groupby(['year', 'month'])['defect_rate_pct'].mean().reset_index()
monthly_defect['period'] = monthly_defect['year'].astype(str) + '-' + monthly_defect['month'].astype(str).str.zfill(2)
monthly_defect = monthly_defect.sort_values('period')

plt.plot(monthly_defect['period'], monthly_defect['defect_rate_pct'], marker='o')
plt.xticks(rotation=90)
plt.title('Monthly Avg Defect Rate %')
plt.ylabel('Defect Rate %')
plt.tight_layout()
plt.show()

### Q26. Which product has the highest total defective units produced?

In [ ]:
prod_prodname = fact_production.merge(dim_product[['product_id', 'product_name']], on='product_id')
top_defects = prod_prodname.groupby('product_name')['defective_units'].sum().nlargest(10)
print(top_defects)

top_defects.plot(kind='bar', title='Top 10 Products by Total Defective Units')
plt.ylabel('Defective Units')
plt.tight_layout()
plt.show()

---
## Section 6 — Shipment Analysis
### Q27. What is the breakdown of shipment statuses?

In [ ]:
status_counts = fact_shipment['status'].value_counts()
print(status_counts)

status_counts.plot(kind='pie', autopct='%1.1f%%', title='Shipment Status Breakdown')
plt.ylabel('')
plt.tight_layout()
plt.show()

### Q28. Which carrier handles the most shipments and has the highest shipping cost?

In [ ]:
carrier_count = fact_shipment['carrier'].value_counts()
carrier_cost  = fact_shipment.groupby('carrier')['shipping_cost'].sum().sort_values(ascending=False)

print('Shipments per Carrier:')
print(carrier_count)
print('\nTotal Shipping Cost per Carrier:')
print(carrier_cost)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
carrier_count.plot(kind='bar', ax=axes[0], title='Shipments per Carrier')
carrier_cost.plot(kind='bar',  ax=axes[1], title='Shipping Cost per Carrier')
plt.tight_layout()
plt.show()

### Q29. What are the most common reasons for delayed shipments?

In [ ]:
delayed = fact_shipment[fact_shipment['status'] == 'Delayed']
delay_reasons = delayed['delay_reason'].value_counts()
print(delay_reasons)

delay_reasons.plot(kind='bar', title='Reasons for Delayed Shipments')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Q30. What is the total shipping cost per customer country?

In [ ]:
ship_cust = fact_shipment.merge(dim_customer[['customer_id', 'country']], on='customer_id')
country_ship_cost = ship_cust.groupby('country')['shipping_cost'].sum().sort_values(ascending=False)
print(country_ship_cost)

country_ship_cost.plot(kind='bar', title='Total Shipping Cost per Customer Country')
plt.ylabel('Shipping Cost USD')
plt.tight_layout()
plt.show()

---
## Section 7 — Cross-Table Analysis
### Q31. What is the quarterly sales performance (revenue & profit) for each year?

In [ ]:
sales_q = fact_sales.merge(dim_date[['date_key', 'year', 'quarter']], on='date_key')
quarterly = sales_q.groupby(['year', 'quarter'])[['net_revenue', 'profit']].sum().round(2)
print(quarterly)

quarterly['net_revenue'].unstack().plot(kind='bar', title='Net Revenue by Quarter & Year')
plt.ylabel('Net Revenue USD')
plt.tight_layout()
plt.show()

### Q32. Is there a correlation between quality_score and lead_time_days in procurement?

In [ ]:
corr = fact_procurement[['quality_score', 'lead_time_days']].corr()
print('Correlation Matrix:')
print(corr)

plt.scatter(fact_procurement['lead_time_days'], fact_procurement['quality_score'], alpha=0.5)
plt.xlabel('Lead Time (days)')
plt.ylabel('Quality Score')
plt.title('Lead Time vs Quality Score')
plt.tight_layout()
plt.show()

### Q33. Which facility type has the highest production output and lowest defect rate?

In [ ]:
prod_factype = fact_production.merge(dim_facility[['facility_id', 'facility_type']], on='facility_id')
factype_summary = prod_factype.groupby('facility_type').agg(
    total_produced=('quantity_produced', 'sum'),
    avg_defect_rate=('defect_rate_pct', 'mean')
).round(2)
print(factype_summary)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
factype_summary['total_produced'].plot(kind='bar', ax=axes[0], title='Total Produced by Facility Type')
factype_summary['avg_defect_rate'].plot(kind='bar', ax=axes[1], title='Avg Defect Rate by Facility Type')
plt.tight_layout()
plt.show()

### Q34. What is the weekend vs weekday sales comparison (revenue & orders)?

In [ ]:
sales_wknd = fact_sales.merge(dim_date[['date_key', 'is_weekend']], on='date_key')
wknd_compare = sales_wknd.groupby('is_weekend').agg(
    total_orders=('sales_id', 'count'),
    total_revenue=('net_revenue', 'sum')
)
wknd_compare.index = ['Weekday', 'Weekend']
print(wknd_compare)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
wknd_compare['total_orders'].plot(kind='bar',   ax=axes[0], title='Orders: Weekday vs Weekend')
wknd_compare['total_revenue'].plot(kind='bar',  ax=axes[1], title='Revenue: Weekday vs Weekend')
plt.tight_layout()
plt.show()

### Q35. Which products have both high sales profit AND high defect rates? (Risk Products)

In [ ]:
# Aggregate profit per product from sales
prod_profit = fact_sales.groupby('product_id')['profit'].sum().reset_index()
prod_profit.columns = ['product_id', 'total_profit']

# Aggregate defect rate per product from production
prod_defect = fact_production.groupby('product_id')['defect_rate_pct'].mean().reset_index()
prod_defect.columns = ['product_id', 'avg_defect_rate']

# Join both and add product name
risk_df = prod_profit.merge(prod_defect, on='product_id')
risk_df  = risk_df.merge(dim_product[['product_id', 'product_name']], on='product_id')

# High profit = above median, high defect = above median
profit_med = risk_df['total_profit'].median()
defect_med = risk_df['avg_defect_rate'].median()

risk_products = risk_df[
    (risk_df['total_profit'] > profit_med) &
    (risk_df['avg_defect_rate'] > defect_med)
].sort_values('total_profit', ascending=False)

print(f'High Profit + High Defect Products: {len(risk_products)}')
print(risk_products[['product_name', 'total_profit', 'avg_defect_rate']].head(10))

plt.scatter(risk_df['avg_defect_rate'], risk_df['total_profit'], alpha=0.6, label='All Products')
plt.scatter(risk_products['avg_defect_rate'], risk_products['total_profit'],
            color='red', alpha=0.8, label='High Risk')
plt.axvline(defect_med, linestyle='--', label='Defect Median')
plt.axhline(profit_med, linestyle='--', label='Profit Median')
plt.xlabel('Avg Defect Rate %')
plt.ylabel('Total Profit USD')
plt.title('Profit vs Defect Rate (Risk Quadrant)')
plt.legend()
plt.tight_layout()
plt.show()